# 03. LightGBM 조화 모델

VIIRS 쪽 특징으로 **MODIS 탐지 수**를 예측합니다. 목적함수는 탐지 "수"에 맞는 **Tweedie**이고, "VIIRS 탐지가 늘면 추정도 는다"는 **단조 제약**을 겁니다(06에서 결정).
횟수 데이터용 목적함수라서,
로그 변환 후 회귀보다 합계를 덜 과소추정합니다. 7일 샘플에서 시험 세트 편향이 -42%에서 -18%로 줄었습니다.
오차 범위는 분위 회귀(10% / 90%)로 추정하고, 결과를 팀에 넘길 `exports/harmonized.parquet` 로 저장합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate  # 한글 폰트 설정 포함
from fsml.paths import default_grid

import lightgbm as lgb

from fsml import gbm
from fsml.baseline import RatioBaseline
from fsml.export import SOURCES, export_harmonized, terra_scale
from fsml.features import make_features, training_rows
from fsml.paths import EXPORTS, MODELS
from fsml.splits import auto_split

GRID = default_grid()
grid = pd.read_parquet(GRID)
res = float(np.round(np.diff(np.sort(grid["lat"].unique())).min(), 6))  # 격자 크기(°)
data, FEATURES = make_features(training_rows(grid), res)
y = data["modis_n"].astype(float)  # 탐지 수 그대로 (Tweedie 목적함수)
kind, (train, valid, test) = auto_split(data)
print(f"{GRID.name} | 격자 {res}° | 분할: {kind} | train {train.sum():,} / valid {valid.sum():,} / test {test.sum():,}")
print("특징:", FEATURES)

## 학습

설정은 `fsml/gbm.py` 의 `PARAMS` 에 있습니다 (04 노트북과 같은 설정).

In [ ]:
model = gbm.fit(data, FEATURES, train, valid)
data["model"] = model.predict(data[FEATURES])

## 기준선과 비교 (시험 세트)

In [ ]:
data["baseline"] = RatioBaseline().fit(data[train | valid]).predict(data)
evaluate.compare(data.loc[test, "modis_n"], {
    "LightGBM": data.loc[test, "model"],
    "기준선": data.loc[test, "baseline"],
    "VIIRS 그대로": data.loc[test, "viirs_n"],
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
evaluate.plot_pred_vs_true(data.loc[test, "modis_n"], data.loc[test, "baseline"], "기준선", axes[0])
evaluate.plot_pred_vs_true(data.loc[test, "modis_n"], data.loc[test, "model"], "LightGBM", axes[1]);

In [ ]:
totals = evaluate.aggregate(data[test], ["model", "baseline"], by=["period"])
evaluate.plot_series(totals, "시험 세트 기간별 합계")
pd.DataFrame({c: evaluate.metrics(totals["modis_n"], totals[c]) for c in ["model", "baseline"]}).T

In [ ]:
imp = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 5), title="특징 중요도 (gain)");

## 오차 범위 (보정된 80% 구간)

분위 회귀(10% / 90%)로 1차 구간을 만든 뒤, 학습에 안 쓴 보정 세트로 구간 폭을 조정합니다(conformal).
그래서 앞으로의 값도 평균적으로 약 80%가 구간 안에 들어오도록 보장됩니다. 자세한 비교는 노트북 06에 있습니다.

In [ ]:
from fsml.intervals import ConformalInterval, coverage

interval = ConformalInterval(coverage=0.8).fit(data, FEATURES, train, valid)
low_t, high_t = interval.predict(data.loc[test, FEATURES])
pd.Series(coverage(data.loc[test, "modis_n"], low_t, high_t))

## 저장 · 내보내기

전체 기간을 **MODIS(Terra + Aqua) 기준 값**으로 내보냅니다. `harmonized`는 칸-기간마다 더 믿을 수 있는 값을 고르고,
`source` 열에 어느 값을 썼는지 남깁니다. 열 형식은 `apps/model/README.md`의 "팀에 넘기는 형식"을 보세요.

| `source` | 기간 | 값 |
| --- | --- | --- |
| `modis` | 2002-07 ~ 2024-12 | 실제 MODIS |
| `terra_scaled` | 2000-11 ~ 2002-06 | Terra만 있던 시기. 10° 블록 × 월별 (Terra + Aqua) ÷ Terra 비율(2003~2019년)을 곱함 |
| `viirs_model` | 2025-01 ~ | VIIRS로 환산한 값. Terra 궤도 변화로 MODIS를 믿을 수 없음(07), Aqua·Terra 종료 이후도 여기에 해당 |

내보낼 모델은 믿을 수 있는 해를 모두 씁니다. 마지막 해로 조기 종료하고, 그 전 해들로 학습합니다.

In [ ]:
year = pd.to_datetime(data["period"]).dt.year.to_numpy()
if len(np.unique(year)) > 1:   # 실제 과거 데이터: 마지막 해로 조기 종료, 그 전 해들로 학습
    final_train, final_valid = year < year.max(), year == year.max()
else:                          # 7일 샘플: 위의 분할 그대로
    final_train, final_valid = train | test, valid
final = gbm.fit(data, FEATURES, final_train, final_valid, log_every=0)
final_interval = ConformalInterval(coverage=0.8).fit(data, FEATURES, final_train, final_valid)
MODELS.mkdir(parents=True, exist_ok=True)
final.save_model(MODELS / "lgbm_harmonize.txt")

full, _ = make_features(grid.reset_index(drop=True), res)
has_viirs = full["viirs_n"].notna().to_numpy()
pred, low, high = (np.full(len(full), np.nan) for _ in range(3))
X = full.loc[has_viirs, FEATURES]
pred[has_viirs] = final.predict(X)
low[has_viirs], high[has_viirs] = final_interval.predict(X)

scale = terra_scale(grid)
out = export_harmonized(full, pred, low, high, EXPORTS / "harmonized.parquet", scale=scale)
print(f"{len(out):,} rows → {EXPORTS / 'harmonized.parquet'}")
print(f"Terra 보정 비율: 10° 블록 × 월 {len(scale):,}개, 중앙값 {scale.median():.2f} (나머지는 위도대 비율)")
out.groupby("source", dropna=False).agg(행=("harmonized", "size"), 합계=("harmonized", "sum"),
                                        시작=("period", "min"), 끝=("period", "max"))

## 다음 실험 (TODO)

- log1p 회귀(`fsml.features.target` / `from_log`)와 비교
  (샘플에선 log1p 회귀가 칸 단위 R²는 약간 높지만 합계를 크게 과소추정했습니다)
- 토지 피복(`landcover`) 열을 붙인 뒤 성능 변화 확인
- `spatial_split` 으로도 검증해, 처음 보는 지역에서의 성능 확인
- 격자 크기(0.25° / 0.5° / 1°)별 성능 비교